# Code for doing the different clustering methods, as well as data preprocessing

In this file we run the different methods

### DMACN - Deep Multi-kernel Auto-encoder Clustering Network
First load the data

In [ ]:
# Example data from article about DMACN - PTSD dataset

from scipy.io import loadmat
import torch

# Load the data
PTSD = loadmat("C:\\Users\\oddar\\Downloads\\PTSD_connectivity.mat")
# PTSD is a dataset containing 87 samples (subjects) with 340 features (as vectorized functional connectivity matrices)
# The expected number of clusters are 3

# Define the functional connectivity matrix (example)
Functional_connectivity_matrix = PTSD["connectivities"]  # Example matrix

PTSD_tensor = torch.from_numpy(Functional_connectivity_matrix).float()

# Dummy data
X = PTSD_tensor  # [N,d] float tensor
print("Data shape:", X.shape)

In [ ]:
# Example data from article about DMACN - AD dataset

from scipy.io import loadmat
import torch

# Load the data
AD = loadmat("Input Data\AD_connectivity.mat")
# AD is a dataset containing 96 samples (subjects) with 888 features (as vectorized functional connectivity matrices)
# The expected number of clusters are 4

# Define the functional connectivity matrix (example)
Functional_connectivity_matrix = AD["connectivities"]  # Example matrix

AD_tensor = torch.from_numpy(Functional_connectivity_matrix).float()

# Dummy data
X = AD_tensor  # [N,d] float tensor
print("Data shape:", X.shape)

In [ ]:
# Loading the data with 29 features (for 29x29 data)
# subject_features.npz contains the vectorized upper triangle of the 29x29 FC matrices, resulting in 50 features per sample.
from scipy.io import loadmat
import torch
from data_loader import load_workable_fc


filepath = "Prosjektoppgave-Odd-Arne-og-Mats-main\\subject_features.npz" # All subjects, 50 features

Functional_connectivity_matrix = load_workable_fc(filepath)

Functional_connectivity_matrix.to_csv("subject_features_clean.csv", index=True)

print(Functional_connectivity_matrix.shape)

X = torch.from_numpy(Functional_connectivity_matrix.values).float()


In [ ]:
# Loading the data with all 200 features (for 200x200 data)
# 200_schaefer_vectorized_fc.mat contains the vectorized upper triangle of the 200x200 FC matrices, resulting in 19900 features per sample.

from scipy.io import loadmat
import torch
import numpy as np

FC_test_mat = loadmat("Input Data\\200_schaefer_vectorized_fc.mat")
# FC_test_mat = loadmat("C:\\Mats og Odd Arne\\Prosjektoppgave\\sch407\\YA\\200_schaefer_vectorized_fc.mat")  # Load the .mat file

FC_test_array = FC_test_mat["200_vectorized_fc"]  # Example matrix
# np.fill_diagonal(FC_test_array, 1.0)  # Set diagonal to zero

print(FC_test_array[0:5].shape)  # Print the first 5 rows to verify

X = torch.from_numpy(FC_test_array).float()  # Example matrix

# Check if any values are abs(X) > 1.0
if torch.any(torch.abs(X) > 1.0):
    print("Warning: Some values in X have absolute value greater than 1.0, which may cause numerical issues in the polynomial kernel.")

In [1]:
#  Loading the data with all 200 features (for 200x200 data) - alternative loading method

from scipy.io import loadmat
import torch
import numpy as np

filepath = "Input Data\FC1.mat"
fc_mat = loadmat(filepath)
fc_mat = fc_mat['FC1']  # Extract the FC1 variable from the loaded .mat file

print(fc_mat.shape)  # Should print (200, 200, N_subjects)

fc_mat_inv = np.transpose(fc_mat, (2, 0, 1))  # Transpose to (N_subjects, 200, 200)
fc_mat_inv = fc_mat_inv[:, None, :, :]  # Add a channel dimension to get (N_subjects, 1, 200, 200)
print(fc_mat_inv.shape)  # Should print (N_subjects, 1, 200, 200)

X = torch.from_numpy(fc_mat_inv).float()  # Convert to PyTorch tensor
print("Data shape:", X.shape)  # Should print (N_subjects, 1, 200, 200)

# Expected shape is (Batch size, channels, height, width)

(200, 200, 72)
(72, 1, 200, 200)
Data shape: torch.Size([72, 1, 200, 200])


In [ ]:
# Loading the data with all 672 features (for 487x672 data) - ADHD dataset

from data_loader import load_static_functional_connectiviies

ADHD = load_static_functional_connectiviies("Input Data\ADHD_connectivity.mat")

Define the autoencoder specs

In [ ]:
kernel_specs = [
    {"kind": "rbf", "t0": 0.01},
    {"kind": "rbf", "t0": 0.05},
    {"kind": "rbf", "t0": 0.1},
    {"kind": "rbf", "t0": 1},
    {"kind": "rbf", "t0": 10},
    {"kind": "rbf", "t0": 50},
    {"kind": "rbf", "t0": 100},
    {"kind": "poly", "a": 0, "b": 2},
    {"kind": "poly", "a": 0, "b": 4},
    {"kind": "poly", "a": 1, "b": 2},
    {"kind": "poly", "a": 1, "b": 4}
]  # h = 3

In [ ]:
# Config for 340x340 data PTSD
from DMACN import DMACN, DMACNConfig


cfg = DMACNConfig(
    name="PTSD_340x340",
    C=3,  # number of clusters
    dims_enc=[340, 285, 240, 202, 170],   # mid = 2 encoder Linear layers = L/2
    dims_dec=[170, 202, 240, 285, 340],
    kernel_specs=kernel_specs,
    m_fuzz=1.08,
    lam1=0.5,
    lam2=0.5,
    lr=1e-3,
    epochs=200,
    mk_max_iters=20,
    mk_eps_stop=1e-5,
    renormalize_omega_sum1=True,
    mid_only_first=True,
    mid_only_last=True,
)

In [ ]:
# Config for 888x888 data AD
from DMACN import DMACN, DMACNConfig


cfg = DMACNConfig(
    name="AD_888x888",
    C=4,  # number of clusters
    dims_enc=[888, 747, 628, 528, 444],   # mid = 2 encoder Linear layers = L/2
    dims_dec=[444, 528, 628, 747, 888],
    kernel_specs=kernel_specs,
    m_fuzz=1.08,
    lam1=1e-2,
    lam2=0.5,
    lr=1e-3,
    epochs=500,
    mk_max_iters=20,
    mk_eps_stop=1e-5,
    renormalize_omega_sum1=True,
    mid_only_first=True,
    mid_only_last=True,
)

In [ ]:
# Config for 29x29 data
from DMACN import DMACN, DMACNConfig


# Config for 29x29 data
cfg = DMACNConfig(
    name = "DMACN_29x29",
    C=3,  # number of clusters
    dims_enc=[29, 25, 22, 19, 17],   # N / 2^(i/l). l=number of layers, N=number of features 
    dims_dec=[17, 19, 22, 25, 29],
    kernel_specs=kernel_specs,
    m_fuzz=1.08,
    lam1=0.5,
    lam2=0.5,
    lr=1e-3,
    epochs=500,
    mk_max_iters=20,
    mk_eps_stop=1e-5,
    renormalize_omega_sum1=True,
    mid_only_first=True,
    mid_only_last=True,
)

In [ ]:
# Config for 19900x19900 data
from DMACN import DMACN, DMACNConfig

# Config for 19900x19900 data
cfg = DMACNConfig(
    C=3,  # number of clusters
    dims_enc=[19900, 15795, 12536, 9950],   # N / 2^(i/l). l=number of layers, N=number of features 
    dims_dec=[9950, 12536, 15795, 19900],
    kernel_specs=kernel_specs,
    m_fuzz=1.08,
    lam1=100,
    lam2=0.5,
    lr=1e-3,
    epochs=200,
    mk_max_iters=200,
    mk_eps_stop=1e-5,
    renormalize_omega_sum1=True,
    mid_only_first=True,
    mid_only_last=True,
)

Config for CAE

In [2]:
# Config for 200x200 data
from Convolutional_AE import DCECConfig
cfg = DCECConfig(
    name="DCEC_200x200",
    n_clusters=3,
    latent_dim=10,
    alpha=1.0,
    gamma=0.1,
    conv_layers_sizes=[1, 32, 64, 128, 256],
    epochs_pretrain=50,
    epochs_dcec=100,
    lr_pretrain=1e-3,
    lr_dcec=1e-3,
    update_interval=10,
    tol=1e-3,
    print_interval=10
)


Run the model

In [ ]:
model = DMACN(cfg, activation="relu")
model.fit(X, verbose_every=10)
labels = model.predict(save=True)
print("labels shape:", labels.shape)
print("labels: ", labels)

torch.save(model.state_dict(), f"model_{cfg.name}.pt") # Saving the model 

In [ ]:
# Evaluate PTSD clustering performance using PTSD_clinical_labels.scv.csv
import pandas as pd
clinical_labels = pd.read_csv("Input Data\PTSD_clinical.scv.csv")
clinical_labels = clinical_labels.iloc[:, 0]  # Assuming the first column contains the labels
true_labels = [0] + clinical_labels.tolist()  # Add a 0 at the beginning to match the number of samples (87)

# Evaluate clustering performance using Adjusted Rand Index (ARI)
from sklearn.metrics import adjusted_rand_score
ari = adjusted_rand_score(true_labels, labels)
print("Adjusted Rand Index (ARI):", ari)

In [ ]:
# Evaluate AD clustering performance using AD_clinical.csv
import pandas as pd
clinical_labels = pd.read_csv("Input Data\AD_clinical.csv")
clinical_labels = clinical_labels.iloc[:, 0]  # Assuming the first column contains the labels
true_labels = [0] + clinical_labels.tolist()  # Add a 0 at the beginning to match the number of samples (87)

# Evaluate clustering performance using Adjusted Rand Index (ARI)
from sklearn.metrics import adjusted_rand_score
ari = adjusted_rand_score(true_labels, labels)
print("Adjusted Rand Index (ARI):", ari)

In [ ]:
# Evaluate ADHD clustering performance using ADHD_clinical.csv
import pandas as pd
import numpy as np
clinical_labels = pd.read_csv("Input Data\ADHD_clinical.csv")
clinical_labels = clinical_labels.iloc[:, 0]  # Assuming the first column contains the labels
true_labels = [0] + clinical_labels.tolist()  # Add a 0 at the beginning to match the number of samples (487)

labels_path = "Clusters\DMACN_ADHD_672x672_3__Clusters__label_0_count_60__label_1_count_237__label_2_count_190.txt"
labels = np.loadtxt(labels_path)

# Evaluate clustering performance using Adjusted Rand Index (ARI)
from sklearn.metrics import adjusted_rand_score, accuracy_score
ari = adjusted_rand_score(true_labels, labels)
print("Adjusted Rand Index (ARI):", ari)
acc = accuracy_score(true_labels, labels)
print("Accuracy built-in:", acc)

from Evaluate_models import evaluate_single_clustering, compute_my_accuracy
acc_custom = compute_my_accuracy(true_labels, labels) # Doing Hungarian accuracy to account for label permutation
print("Accuracy custom:", acc_custom)
evaluate_single_clustering(X, labels)
y_mid = model.y_mid.cpu().detach().numpy()
evaluate_single_clustering(y_mid, labels)

In [ ]:
# Evaluate the results simply
from Evaluate_models import evaluate_single_clustering
evaluate_single_clustering(X, labels)
y_mid = model.y_mid.cpu().detach().numpy()
evaluate_single_clustering(y_mid, labels)

### CAE / DECE

In [ ]:
from Convolutional_AE import DCEC, pretrain_cae, initialize_cluster_centers, train_dcec, predict_soft_assignments
from torch.utils.data import TensorDataset, DataLoader

# ----------------------------------------------------------
# DCEC - Deep Convolutional Embedded Clustering
# ----------------------------------------------------------

dataset = TensorDataset(X)
dataloader = DataLoader(dataset, batch_size=16, shuffle=False)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# TODO: run the model for multiple cluster numbers and evaluate the clustering performance using Calinski-Harabasz index and silhouette score

model = DCEC(cfg=cfg).to(device)

pretrain_cae(model, dataloader, device, epochs=cfg.epochs_pretrain, lr=cfg.lr_pretrain, print_interval=cfg.print_interval)
y_pred_initial = initialize_cluster_centers(model, dataloader, device)
print("Early cluster centers initialized.", y_pred_initial)

train_dcec(
    model,
    dataloader,
    device,
    gamma=cfg.gamma,
    epochs=cfg.epochs_dcec,
    lr=cfg.lr_dcec,
    update_interval=cfg.update_interval,
    tol=cfg.tol,
    print_interval=cfg.print_interval
    )

q_final, labels = predict_soft_assignments(model, dataloader, device, save=True)
save_z(model, model.z)
print("Predicted cluster labels:", labels)

# Saving the model
torch.save(model.state_dict(), f"Models\\model_{cfg.name}.pt") 

In [ ]:
# Evaluate clustering performance of 200x200 matrix
from Evaluate_models import evaluate_single_clustering

triu_idx = np.triu_indices(200, k=1)
fc_mat_transp = np.transpose(fc_mat, (2, 0, 1))  # Transpose to (N_subjects, 200, 200)

Features = np.array([
    fc_mat_transp[i][triu_idx] for i in range(fc_mat_transp.shape[0])
]) 
predictions = labels

z = model.z
print("z shape:", z.shape)
print("predictions shape:", predictions.shape)

evaluate_single_clustering(Features, predictions)
evaluate_single_clustering(z, predictions)


Run multiple models after one-another

In [ ]:
from Convolutional_AE import DCEC, pretrain_cae, initialize_cluster_centers, train_dcec, predict_soft_assignments, save_z
from torch.utils.data import TensorDataset, DataLoader

# ----------------------------------------------------------
# DCEC - Deep Convolutional Embedded Clustering
# ----------------------------------------------------------

dataset = TensorDataset(X)
dataloader = DataLoader(dataset, batch_size=16, shuffle=False)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# TODO: run the model for multiple cluster numbers and evaluate the clustering performance using Calinski-Harabasz index and silhouette score

cluster_sizes =  list(range(2, 11)) # Example cluster sizes to evaluate

temp_name = cfg.name  # Store the original name to modify it for each cluster size

for n_clusters in cluster_sizes:
    cfg.n_clusters = n_clusters
    cfg.name = f"{temp_name}_clusters_{n_clusters}"
    
    model = DCEC(cfg=cfg).to(device)

    pretrain_cae(model, dataloader, device, epochs=cfg.epochs_pretrain, lr=cfg.lr_pretrain, print_interval=cfg.print_interval)
    y_pred_initial = initialize_cluster_centers(model, dataloader, device)
    print("Early cluster centers initialized.", y_pred_initial)

    train_dcec(
        model,
        dataloader,
        device,
        gamma=cfg.gamma,
        epochs=cfg.epochs_dcec,
        lr=cfg.lr_dcec,
        update_interval=cfg.update_interval,
        tol=cfg.tol,
        print_interval=cfg.print_interval
        )

    q_final, labels = predict_soft_assignments(model, dataloader, device, save=True)
    save_z(model, model.z)
    print("Predicted cluster labels:", labels)

    # Saving the model
    torch.save(model.state_dict(), f"Models\\model_{cfg.name}.pt") 


In [14]:
# Evaluate clustering performance of 200x200 matrix
from Evaluate_models import evaluate_single_clustering
import os
import numpy as np

# Load the middle layer and predicted labels from the saved model
# Open the folder "Models"
for model in os.listdir("Clusters"):
    # If the model name begins with "model_DCEC" load it
    if model.startswith("DCEC_200x200_cluster") & ("_labels_predicted_labels_" in model):
        print(f"\n Evaluating model: {model}")

        labels_path = os.path.join("Clusters", model)  # Path to the saved labels
        # Z path is the same as labels path but with "labels" replaced by "middle_layer"
        z_path = labels_path.replace("labels_predicted_labels_", "middle_layer_predicted_labels_")  # Path to the saved middle layer
        
        labels = np.loadtxt(labels_path, dtype=object)  # Load the saved labels
        z = np.loadtxt(z_path, dtype=object)  # Load the saved middle layer

        evaluate_single_clustering(z, labels)


 Evaluating model: DCEC_200x200_cluster_10_labels_predicted_labels_label_0_3_label_2_21_label_3_2_label_4_7_label_6_25_label_7_1_label_8_5_label_9_8.txt

 Scores for the given labels:
Silhouette coefficient: 0.6960086590411048
Davies-Bouldin score: 0.37986792455990614
Calinski-Harabasz score: 2300.075291307419

 Evaluating model: DCEC_200x200_cluster_2_labels_predicted_labels_label_0_32_label_1_40.txt

 Scores for the given labels:
Silhouette coefficient: 0.7453915594673854
Davies-Bouldin score: 0.32943377272271424
Calinski-Harabasz score: 410.6027639505361

 Evaluating model: DCEC_200x200_cluster_3_labels_predicted_labels_label_0_20_label_1_31_label_2_21.txt

 Scores for the given labels:
Silhouette coefficient: 0.8029290216533496
Davies-Bouldin score: 0.31477099176958173
Calinski-Harabasz score: 813.992910852845

 Evaluating model: DCEC_200x200_cluster_4_labels_predicted_labels_label_2_72.txt
Only one cluster in the provided labels, silhouette coefficient not computed.

 Evaluating 

In [3]:
# Load the models and run predict_soft_assignments 
from Convolutional_AE import predict_soft_assignments, DCEC
from torch.utils.data import TensorDataset, DataLoader
import torch

dataset = TensorDataset(X)
dataloader = DataLoader(dataset, batch_size=16, shuffle=False)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

cluster_sizes =  list(range(2, 11)) # Example cluster sizes to evaluate

temp_name = cfg.name  # Store the original name to modify it for each cluster size

for cluster_number in cluster_sizes: 
    cfg.n_clusters = cluster_number
    
    model = DCEC(cfg=cfg).to(device)
    model.load_state_dict(torch.load(f"Models\\model_DCEC_200x200_clusters_{cluster_number}.pt"))
    predict_soft_assignments(model, dataloader, device, save=True)

Predicted labels shape: (72,)
Embeddings shape: (72, 10)
Predicted labels shape: (72,)
Embeddings shape: (72, 10)
Predicted labels shape: (72,)
Embeddings shape: (72, 10)
Predicted labels shape: (72,)
Embeddings shape: (72, 10)
Predicted labels shape: (72,)
Embeddings shape: (72, 10)
Predicted labels shape: (72,)
Embeddings shape: (72, 10)
Predicted labels shape: (72,)
Embeddings shape: (72, 10)
Predicted labels shape: (72,)
Embeddings shape: (72, 10)
Predicted labels shape: (72,)
Embeddings shape: (72, 10)


In [ ]:
# Code for visualising the clustering results 
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt



### UMAP

In [ ]:
from UMAP import UMAP

### HDBSCAN
Perform HDBSCAN on the data

In [ ]:
from HDBSCAN import hdbscan_clustering

hdbscan_clustering(Functional_connectivity_matrix=Functional_connectivity_matrix, save_labels=True)

### Evaluate the clusters
Evaluate the clusters using simple methods: Silhouette coefficient, Davies-Bouldin score and Calinski-Harabasz score

In [ ]:
from Evaluate_models import evaluate_clustering, evaluate_single_clustering
import os

# Define where to find the labels 
labels_path = "Clusters\DMACN__Clusters_3__label_0_21_label_1_19_label_2_32.txt"
evaluate_single_clustering(functional_connectivity_matrix=X, labels_path=labels_path)